## This notebook is related to open ai function calling

# OpenAI Function Calling Tutorial
Learn how to extend ChatGPT's capabilities with custom functions


## What is Function Calling?
- Allows ChatGPT to call external functions/APIs
- Converts natural language requests into structured function calls
- Enables ChatGPT to interact with real-world systems



## What we'll cover:
1. Basic function calling concepts
2. Function definition and schema
3. Simple examples (calculator, weather)
4. Advanced examples (database queries, API calls)
5. Best practices and error handling
6. Real-world applications

In [5]:
import openai
import os

from dotenv import load_dotenv

load_dotenv()



True

In [6]:
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [9]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that can answer questions and help with tasks."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)

The capital of France is Paris.


In [10]:
def get_open_ai_response(prompt):
    response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are specilized in GEN so anser the question like a GEN AI specilist"},
        {"role": "user", "content": prompt}
    ]
    )

    print("Traditional Response:")
    print(response.choices[0].message.content)


In [11]:
get_open_ai_response("answer me 2+2*75")

Traditional Response:
The answer to 2 + 2 * 75 is 152.


In [ ]:
# Define a simple calculator function
### Just think it is a agent that can do the calculation


def calculate(operation: str, a: float, b: float) -> float:
    """
    Perform basic mathematical operations
    
    Args:
        operation: The operation to perform (+, -, *, /)
        a: First number
        b: Second number
    
    Returns:
        Result of the calculation
    """
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        if b == 0:
            return "Error: Division by zero"
        return a / b
    else:
        return "Error: Invalid operation"

result = calculate("*", 15, 23)

In [13]:
result

345

In [14]:
import json

# Define the function schema for openai

calculate_function = {
    "name": "calculate",
    "description": "Perform basic mathematical operations",
    "parameters": {
        "type": "object",
        "properties": {
            "operation": {"type": "string", "enum": ["+", "-", "*", "/"]},
            "a": {
                "type": "number"
                },
            "b": {"type": "number"}
        },
        "required": ["operation", "a", "b"]
    }
}


print(json.dumps(calculate_function, indent=2))

{
  "name": "calculate",
  "description": "Perform basic mathematical operations",
  "parameters": {
    "type": "object",
    "properties": {
      "operation": {
        "type": "string",
        "enum": [
          "+",
          "-",
          "*",
          "/"
        ]
      },
      "a": {
        "type": "number"
      },
      "b": {
        "type": "number"
      }
    },
    "required": [
      "operation",
      "a",
      "b"
    ]
  }
}


In [20]:
def chat_with_calculator(prompt):
    """
    Chat with OpenAI using function calling for calculations
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant that can perform calculations."},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        functions=[calculate_function],
        function_call="auto"
    )

    message = response.choices[0].message

    if message.function_call:
        function_name = message.function_call.name
        function_args = json.loads(message.function_call.arguments)

        if function_name == "calculate":
            result = calculate(**function_args)
            return f"The result of {function_args['operation']} {function_args['a']} and {function_args['b']} is {result}"
    else:
        return message.content


In [21]:
chat_with_calculator("who is the PM of India")

"I'm a helpful assistant that can perform calculations. If you have any numbers that you'd like me to perform calculations on, feel free to let me know!"

In [22]:
chat_with_calculator("what is 2+2*75")

'The result of + 2 and 150 is 152'

In [23]:
result = chat_with_calculator("What's 15 * 23 + 45? Please calculate step by step.")

In [24]:
result

'The result of * 15 and 23 is 345'

In [25]:
def get_weather(city: str, country: str = "US") -> str:
    """
    Get current weather for a city
    
    Args:
        city: Name of the city
        country: Country code (default: US)
    
    Returns:
        Weather information as string
    """

    mock_weather_data = {
        "new york": {"temp": 22, "condition": "sunny", "humidity": 60},
        "london": {"temp": 15, "condition": "cloudy", "humidity": 80},
        "tokyo": {"temp": 28, "condition": "rainy", "humidity": 75}
    }

    city_key = city.lower()
    if city_key in mock_weather_data:
        data = mock_weather_data[city_key]
        return f"Weather in {city}: {data['temp']}°C, {data['condition']}, humidity: {data['humidity']}%"
    else:
        return f"Weather data not available for {city}"


        



In [26]:
weather_function = {
    "name": "get_weather",
    "description": "Get current weather information for a specific city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Name of the city"
            },
            "country": {
                "type": "string",
                "description": "Country code (e.g., US, UK, JP)",
                "default": "US"
            }
        },
        "required": ["city"]
    }
}

###  Multiple Functions Handler

In [27]:
class FunctionHandler:
    def __init__(self):
        self.functions = {
            "calculate": calculate,
            "get_weather": get_weather
        }

        self.function_schema = [
            calculate_function,
            weather_function
        ]

    def call_function(self, function_name, arguments):
        """Call the appropriate function with given arguments"""

        if function_name in self.functions:
            return self.functions[function_name](**arguments)
        else:
            return f"Function {function_name} not found"


    def chat(self, user_message):

        """Enhanced chat with multiple function support"""

        messages = [{"role": "user", "content": user_message}]

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            functions=self.function_schema,
            function_call="auto"
        )

        message = response.choices[0].message

        if message.function_call:
            function_name = message.function_call.name
            arguments = json.loads(message.function_call.arguments)

            result = self.call_function(function_name, arguments)

                
            messages.extend([
                {
                    "role": "assistant",
                    "content": None,
                    "function_call": {
                        "name": function_name,
                        "arguments": json.dumps(arguments)
                    }
                },
                {
                    "role": "function",
                    "name": function_name,
                    "content": str(result)
                }
            ])

            final_response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages
            )
            return final_response.choices[0].message.content
        return message.content

        
            
            
         





        

In [28]:
handler = FunctionHandler()

result = handler.chat("What's the weather in Tokyo?")
result




'The weather in Tokyo is currently 28°C with rain and a humidity level of 75%.'

In [29]:
handler.chat("what is the result of 2+2")

'The result of 2 + 2 is 4.'

In [30]:
handler.chat("What's the weather in Tokyo and what is the result of 2+2")

'The weather in Tokyo is currently 28°C with rain and 75% humidity. \n\nAs for the result of 2+2, it is 4.'

### Limitations

1. It won't able to do the to from and communications
2. it won't able to provide resoning
3. it won't able to scale